# Előkészítés — helyi futtatás (Windows + VS Code)

**MI alapú ember-gép interakció · Emberi emóciók felismerése**

Ez a notebook a **saját gépeden** készíti elő a gyakorlatot. Egyszer kell
lefuttatni; utána az adat és a modell a gépeden marad.

Mit csinál:

1. ellenőrzi a Python-környezetet és a GPU-t
2. megnézi, minden csomag megvan-e
3. betölti a segédmodult (`ser_utils.py` — a notebook mellett legyen)
4. letölti a RAVDESS adatbázist a `ser_gyakorlat` mappába (~208 MB)
5. letölti a WavLM modellt (~380 MB, a felhasználói gyorsítótárba)
6. lefuttat egy rövid öntesztet

> **A környezetet előbb létre kell hozni** — lásd a `TELEPITES_WINDOWS.md`-t.
> Röviden, Anaconda Promptban:
>
> ```
> conda create -n ser python=3.11 -y
> conda activate ser
> pip install torch --index-url https://download.pytorch.org/whl/cu126
> pip install transformers datasets soundfile librosa opensmile ^
>             scikit-learn scikit-image pandas matplotlib sounddevice ipykernel
> ```
>
> VS Code-ban jobbra fent válaszd ki kernelnek a `ser` környezetet.

## 1 · Futtatókörnyezet ellenőrzése

## 1 · Környezet és GPU ellenőrzése

In [1]:
import sys
print("Python:   ", sys.version.split()[0])
print("futtatható:", sys.executable)

try:
    import torch
    print("torch:    ", torch.__version__, "| CUDA elérhető:", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"GPU:       {p.name} · {p.total_memory/1024**3:.1f} GB VRAM")
    else:
        print("\nA torch CPU-s változata van telepítve, vagy nincs CUDA-illesztő.")
        print("CPU-n is működik minden, csak az embedding-számítás lassabb.")
        print("GPU-s torch: pip install torch --index-url "
              "https://download.pytorch.org/whl/cu126")
except ImportError:
    print("\nA torch nincs telepítve — lásd a TELEPITES_WINDOWS.md-t.")

Python:    3.11.16
futtatható: c:\Users\velox\miniconda3\envs\ser\python.exe
torch:     2.14.0+cu132 | CUDA elérhető: True
GPU:       NVIDIA GeForce RTX 4050 Laptop GPU · 6.0 GB VRAM


## 2 · Csomagok telepítése

Ez tart a legtovább, kb. 2-3 perc.

## 2 · Csomagok ellenőrzése

Ha valami hiányzik, telepítsd az **Anaconda Promptban** (nem itt), aktivált `ser` környezetben, majd indítsd újra a kernelt.

In [2]:
import importlib

kell = ["numpy", "scipy", "sklearn", "soundfile", "librosa", "pandas",
        "matplotlib", "torch", "transformers", "opensmile", "sounddevice",
        "datasets", "skimage"]
hianyzik = []
for m in kell:
    try:
        importlib.import_module(m)
    except ImportError:
        hianyzik.append(m)

if hianyzik:
    print("HIÁNYZÓ CSOMAG:", ", ".join(hianyzik))
    print("\nTelepítés (Anaconda Prompt, aktivált ser környezetben):")
    print("   pip install " + " ".join(hianyzik))
    print("\nUtána: VS Code → Restart kernel")
else:
    print("Minden csomag megvan.")

c:\Users\velox\miniconda3\envs\ser\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HIÁNYZÓ CSOMAG: datasets, skimage

Telepítés (Anaconda Prompt, aktivált ser környezetben):
   pip install datasets skimage

Utána: VS Code → Restart kernel


## 3 · A segédmodul betöltése

A `ser_utils.py` tartalmazza a letöltő, jellemzőkinyerő és kiértékelő függvényeket. A notebook **mellett** legyen.

In [3]:
# --- a segédmodul betöltése (helyi futtatás) -----------------------------
# A ser_utils.py a notebook MELLETT legyen, ugyanabban a mappában.
import os, sys, importlib

mappa = os.getcwd()
if mappa not in sys.path:
    sys.path.insert(0, mappa)

fajl = os.path.join(mappa, "ser_utils.py")
if not os.path.exists(fajl):
    raise RuntimeError(
        f"Nem találom a ser_utils.py-t itt: {mappa}\n"
        "Tedd a notebook mellé, és futtasd újra ezt a cellát.\n"
        "(VS Code-ban a notebook munkakönyvtára alapértelmezésben a notebook mappája.)")
if os.path.getsize(fajl) < 1000 or "def gpu_info" not in open(fajl, encoding="utf-8").read():
    raise RuntimeError(f"A ser_utils.py ({os.path.getsize(fajl)} bájt) üres vagy csonka.")

sys.modules.pop("ser_utils", None)      # esetleges korábbi, hibás import eldobása
import ser_utils as su
su.self_check()

ser_utils v1.5
  függvények: 11/11 megvan
  GPU: NVIDIA GeForce RTX 4050 Laptop GPU · 6.0 GB VRAM · CUDA 13.2
  RENDBEN


True

## 4 · Az adat letöltése

A korpusz a notebook melletti `ser_gyakorlat` mappába kerül (~208 MB). Ha már megvan, nem tölti le újra.

In [4]:
import os

WORK = os.path.abspath("ser_gyakorlat")
os.makedirs(WORK, exist_ok=True)
print("munkakönyvtár:", WORK)

wav_root = su.download_ravdess(WORK)

munkakönyvtár: d:\Work\bme\vitmma23\gyakorlat_text_image\ser_gyakorlat
Letöltés (Audio_Speech_Actors_01-24.zip, ~208 MB)…
  100%  (   208 / 208 MB)
Kicsomagolás…
Kész: 1440 felvétel a(z) d:\Work\bme\vitmma23\gyakorlat_text_image\ser_gyakorlat\ravdess könyvtárban.


## 5 · A modell letöltése

A WavLM súlyai a felhasználói gyorsítótárba kerülnek (`C:\\Users\\<név>\\.cache\\huggingface`), és ott is maradnak.

In [5]:
from transformers import AutoFeatureExtractor, AutoModel
MODEL = "microsoft/wavlm-base-plus"
_ = AutoFeatureExtractor.from_pretrained(MODEL)
_ = AutoModel.from_pretrained(MODEL)
print("A modell a gyorsítótárban van.")

Loading weights: 100%|██████████| 248/248 [00:00<00:00, 36041.28it/s]

A modell a gyorsítótárban van.


## 6 · Önteszt

Ha ez a cella hibátlanul lefut, órán minden működni fog.

In [6]:
import numpy as np

df = su.scan_ravdess(wav_root)
print("felvételek:", len(df), "| színészek:", df.actor.nunique(),
      "| érzelmek:", df.emotion.nunique())

# 8 felvétel végigfuttatása a teljes láncon
mini = df.sample(8, random_state=0)
waves = su.load_all(mini.path.tolist(), show_progress=False)
Xh, names = su.egemaps_features(waves, show_progress=False)
E = su.wavlm_embeddings(waves, layers=6, show_progress=False)

print(f"kézi jellemzők:  {Xh.shape}")
print(f"WavLM embedding: {E.shape}")
print("\nMINDEN RENDBEN — készen állsz a gyakorlatra.")

felvételek: 1440 | színészek: 24 | érzelmek: 8


Loading weights: 100%|██████████| 248/248 [00:00<00:00, 17778.25it/s]
c:\Users\velox\miniconda3\envs\ser\Lib\site-packages\torch\nn\functional.py:6829: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


kézi jellemzők:  (8, 88)
WavLM embedding: (8, 768)

MINDEN RENDBEN — készen állsz a gyakorlatra.


---

### Ha valami nem működött

| tünet | mit tegyél |
|---|---|
| `ModuleNotFoundError` | rossz kernel van kiválasztva. VS Code jobb felső sarok → kernel → a `ser` környezet |
| `torch.cuda.is_available()` hamis | CPU-s torch van fent. `pip uninstall torch`, majd a CUDA-s index-url-lel újra |
| `AttributeError: module 'ser_utils' has no attribute ...` | a `ser_utils.py` üres vagy csonka, vagy nem a notebook mellett van |
| `Nem találom a ser_utils.py-t` | VS Code-ban a notebook munkakönyvtára nem a notebook mappája — futtasd: `import os; os.chdir(r"<a notebook mappája>")` |
| a letöltés megszakad | futtasd újra a 4. cellát; a csonka zip-et magától eldobja |
| `sounddevice` hiba a felvételnél | `pip install sounddevice`; ha nem a jó mikrofon szól bele: `su.list_microphones()` |
| CUDA out of memory | a modul magától felezi a batch méretet; ha így sem megy, add meg: `su.wavlm_embeddings(..., device="cpu")` |